# TP n°7 — Cryptographie asymétrique  

## Objectifs
Comprendre les principes de la cryptographie asymétrique.

## Exercice 1 : Boîte à outils
Vous devez constituer une petite bibliothèque de calcul modulo un nombre premier.

**Q1.** Implémentez l’inversion modulo \(p\) (via l’algorithme d’Euclide étendu).

**Q2.** Implémentez un crible pour construire la liste des nombres premiers inférieurs à une certaine borne.  
L’algorithme classique est le suivant : partir d’un tableau de taille \(n\), et marquer les cases multiples de \(i\) où \(i > 1\) est le numéro de la plus petite case non marquée. À la fin, les cases non marquées sont aux positions premières.  
Générez la liste des premiers jusqu’à \(1\,000\,000\) et stockez-la dans un fichier.

**Q3.** Implémentez un test de primalité.

**Q4.** Implémentez le calcul des diviseurs premiers d’un entier donné.

**Q5.** Implémentez le calcul d’un générateur de \((\mathbb{Z}/p\mathbb{Z})^\*\). Vous pouvez tirer un élément au hasard tant que vous ne trouvez pas de générateur.

**Q6.** Implémentez l’exponentiation modulo un nombre premier.

In [ ]:
from random import random


def euclide_etendu(a, b):
    if b == 0:
        return a, 1, 0
    pgcd, u1, v1 = euclide_etendu(b, a % b)
    u = v1
    v = u1 - (a // b) * v1
    return pgcd, u, v

def inverse_mod(a, p):
    pgcd, x, _ = euclide_etendu(a % p, p)
    if pgcd != 1:
        raise ValueError(f"{a} pas inversible modulo {p}")
    return x % p

def crible(n):
    p = [True]*(n+1) #ou n ?
    p[0] = False
    p[1] = False

    #on commence à i=2
    i=2
    while i*i <= n : #on s'arrête à la racine de n, car après la liste des diviseurs est symétrique
        if p[i]:
            for j in range(i*i, n+1, i):
                p[j] = False
        i+=1

    premiers = [i for i in range(n+1) if p[i]]
    return premiers

def est_premier(n):
    if n<2:
        return False
    if n==2:
        return True
    if n%2==0:
        return False

    i = 3
    while i*i <= n:
        if n%i == 0:
            return False
        i+=2 #on saute les pairs
    return True

def diviseur_premier(n):
    diviseurs = []

    if n%2==0:
        diviseurs.append(2)
        while n%2==0:
            n = n//2

    i=3
    while i*i <= n:
        if n%i == 0:
            diviseurs.append(i)
            while n%i == 0:
                n = n//i
        i+=2

    if n>1:
        diviseurs.append(n)
    return diviseurs

def est_generateur(g, p, fact_premier):
    """Verifie si g est un générateur de Z/pZ*"""
    for q in fact_premier: #où fact_premier est les facteurs premiers de p-1
        if pow(g, (p-1)//q, p)==1:
            return False
    return True

def find_gen(p):
    """Trouve un générateur de Z/pZ* par tirage aléatoire"""
    facteurs = diviseur_premier(p-1)
    while True:
        g=random.randint(2,p-1)
        if est_generateur(g, p, facteurs):
            return g

def exp_mod(g, x,p):
    """calcul efficace de g^x mod p"""

    result = 1
    g = g%p #pour le cas où g>=p

    while x>0:
        if x%2==1:
            result = (result * g) % p
        g=(g*g)%p
        x=x//2

    return result

## Exercice 2 : Protocole de Diffie-Hellman

**Q1.** Implémentez une communication entre *Arielle* et *Bertrand* sous TCP permettant d’appliquer le protocole
de Diffie-Hellman. Vous pouvez considérer que l’un est un **serveur** et l’autre un **client**.  

Chaque participant doit afficher les étapes du protocole sur sa sortie standard.

In [ ]:
import threading
import socket as sckt
import random
import time

# Public Diffie-Hellman parameters
p = 67   # prime number
g = 5    # generator modulo p


def bertrand(port):
    def bertrand_behavior(port):
        # Bertrand chooses private key b
        b = random.randint(2, p - 2)

        sock = sckt.socket(sckt.AF_INET, sckt.SOCK_STREAM)
        sock.bind(("127.0.0.1", port))
        sock.listen(1)

        print(f"Bertrand : initialized on port {port}")
        print(f"Bertrand : public parameters p = {p}, g = {g}")
        print(f"Bertrand : private key b = {b}")
        print("Bertrand : Listening")

        conn, addr = sock.accept()
        print(f"Bertrand : Got new connection from {addr[0]} at {addr[1]}")

        # Bertrand waits for Arielle's public value g^a mod p
        ga = int(conn.recv(1024).decode())
        print(f"Bertrand : Received g^a mod p = {ga} from {addr[1]}")

        # Bertrand computes and sends g^b mod p
        gb = pow(g, b, p)
        print(f"Bertrand : computed g^b mod p = {gb}")
        print(f"Bertrand : Sending {gb} to {addr[1]}")
        conn.send(str(gb).encode())

        # Bertrand computes shared key: (g^a)^b mod p
        gab = pow(ga, b, p)
        print(f"Bertrand : calculated shared key = {gab}")

        conn.close()
        print(f"Bertrand : Closing the connection from {addr[0]} at {addr[1]}")
        sock.close()
        print("Bertrand : Disconnected")

    x = threading.Thread(target=bertrand_behavior, args=(port,))
    x.start()
    return x


def arielle(port):
    # Arielle chooses private key a
    a = random.randint(2, p - 2)

    sock = sckt.socket(sckt.AF_INET, sckt.SOCK_STREAM)

    try:
        sock.connect(("127.0.0.1", port))
        print(f"Arielle {sock.getsockname()[1]} : connected\n")
        print(f"Arielle : public parameters p = {p}, g = {g}")
        print(f"Arielle : private key a = {a}")
    except:
        print("Arielle : Connection Error")
        raise Exception

    # Arielle computes and sends g^a mod p
    ga = pow(g, a, p)
    print(f"Arielle : computed g^a mod p = {ga}")
    print(f"Arielle : Sending {ga} to {port}")
    sock.send(str(ga).encode())

    # Arielle waits for Bertrand's public value g^b mod p
    gb = int(sock.recv(1024).decode())
    print(f"Arielle : Received g^b mod p = {gb} from {port}")

    # Arielle computes shared key: (g^b)^a mod p
    gab = pow(gb, a, p)
    print(f"Arielle : calculated shared key = {gab}")

    sock.close()
    print("Arielle : Disconnected")


port = 1024

server_thread = bertrand(port)
time.sleep(0.5)   # give the server time to start
arielle(port)
server_thread.join()

## Exercice 3 : Attaque par l’homme du milieu
On se place maintenant dans le cadre d’une attaque par l’homme du milieu. Pour ne pas rentrer dans les
détails techniques de comment l’attaquant peut intercepter les communications, considérez que les clients,
qui sont sur les machines d’Arielle et Bertrand, communiquent avec un serveur sur la machine de l’attaquant
Laurent. Arielle et Bertrand communique en utilisant RSA.

**Q1.** Implémentez le serveur de Laurent pour qu’il se fasse passer pour Bertrand auprès d’Arielle (et inversement).
Laurent affiche sur la sortie standard les messages échangés par Arielle et Bertrand sous forme déchiffrée.

In [ ]:
import socket as sckt
import threading
cbl = crible(1_000_000)
def generate_rsa_keypair():
    # Prendre deux grands nombres premiers de la liste
    primes = [p for p in cbl if p > 1000]
    p = random.choice(primes)
    q = random.choice(primes)
    while p == q:
        q = random.choice(primes)

    n = p * q
    phi = (p - 1) * (q - 1)

    e = 3
    while True:
        try:
            d = inverse_mod(e, phi)
            break
        except ValueError:
            e += 2

    return (e, n), (d, n)

def rsa_encrypt(m, e, n):
    return pow(m, e, n)

def rsa_decrypt(c, d, n):
    return pow(c, d, n)

def arielle(port):
    # connexion à Laurent

    sock = sckt.socket(sckt.AF_INET, sckt.SOCK_STREAM)
    sock.connect(("127.0.0.1", port))
    # génération des clés + échange des clés publiques
    data = sock.recv(1024).decode()
    e_recv, n_recv = map(int, data.split(","))

    # communication
    (e_a, n_a), _ = generate_rsa_keypair()
    sock.send(f"{e_a},{n_a}".encode())
    message = 42
    c = rsa_encrypt(message, e_recv, n_recv)
    sock.send(str(c).encode())
    print(f"[Arielle] Message envoyé : {message}")
    sock.close()

def bertrand(port):
    # connexion à Laurent
    def run(port):

        (e_b, n_b), (d_b, _) = generate_rsa_keypair()
        sock = sckt.socket(sckt.AF_INET, sckt.SOCK_STREAM)
        sock.bind(("127.0.0.1", port))
        sock.listen(1)
        conn, _ = sock.accept()
        conn.send(f"{e_b},{n_b}".encode())
    # génération des clés + échange des clés publiques

    # communication
        conn.recv(1024)
        c = int(conn.recv(1024).decode())
        m = rsa_decrypt(c, d_b, n_b)
        print(f"[Bertrand] Message reçu déchiffré : {m}")
        conn.close(); sock.close()
    threading.Thread(target=run, args=[port]).start()
def laurent(port_a, port_b):
    def run(port_a,port_b):

        (e_l, n_l), (d_l, _) = generate_rsa_keypair()
        #connexion a arielle et bertrand

        srv = sckt.socket(sckt.AF_INET, sckt.SOCK_STREAM)
        #on écoute arielle
        srv.bind(("127.0.0.1", port_a))
        srv.listen(1)
        #pour bertrand
        sock_b = sckt.socket(sckt.AF_INET, sckt.SOCK_STREAM)
        sock_b.connect(("127.0.0.1", port_b))
        data = sock_b.recv(1024).decode()
        e_b, n_b = map(int, data.split(","))
        #arielle se connecte à bertrand, et laurant lui envoi sa clé
        conn_a, _ = srv.accept()
        conn_a.send(f"{e_l},{n_l}".encode())
        conn_a.recv(1024)

        #se fait passer pour arielle vers bertrand

        sock_b.send(f"{e_l},{n_l}".encode())
        c_a = int(conn_a.recv(1024).decode())

        m = rsa_decrypt(c_a, d_l, n_l)
        print(f"[Laurent] Message intercepté en clair : {m}")

        c_b = rsa_encrypt(m, e_b, n_b)
        sock_b.send(str(c_b).encode())

        conn_a.close(); sock_b.close(); srv.close()

    threading.Thread(target=run, args=[port_a, port_b]).start()
#on test pour voir si laurent récupère bien les messages en clair

import time
port_bertrand = 1024
port_laurent  = 1025

bertrand(port_bertrand)
time.sleep(0.1)
laurent(port_laurent, port_bertrand)
time.sleep(0.2)
arielle(port_laurent)

## Exercice 4 : Malléabilité et signature

Laurent n’a pas eu le temps de mettre en place sa stratégie d’attaque par homme du milieu, mais il peut cependant intercepter les messages transmis. Il ne peut juste pas les déchiffrer. Laurent a accès aux clés publiques RSA d’Arielle et Bertrand, ainsi qu’aux messages chiffrés, qu’il peut intercepter, modifier et retransmettre.

**Q1.** Implémentez Laurent pour que, lorsqu’il intercepte un message chiffré c correspondant au message m (qu’il ne peut retrouver), il transmette à l’autre participant un message c′ correspondant au message 2m.


#### Principe mathématique de l’attaque :

L'idée est la suivante : Notons $e, n$ la clé publique de Bertrand, et $M$ le message envoyé par Arielle. Le message chiffré intercepté par Laurent est alors $C = M^e \mod n$.
On veut envoyer un message $C' = (2M)^e \mod n$.

Or : 

$$
\begin{align}
C' &= (2M)^e \mod n \\
&= 2^e M^e \mod n \\
&= (2^e \mod n) (M^e \mod n) \mod n \\
&= (2^e \mod n) C \mod n
\end{align}
$$

Avec Laurent, on doit donc juste calculer $l = 2^e \mod n$, puis renvoyer à Bertrand le message $C' = (l \cdot C) \mod n$.

In [ ]:
import random
from utils import *
import socket
import pickle

PORT = 11112

cbl = crible(1_000_000)

def generate_rsa_keypair():
    # Prendre deux grands nombres premiers de la liste
    primes = [p for p in cbl if p > 1000]
    p = random.choice(primes)
    q = random.choice(primes)
    while p == q:
        q = random.choice(primes)
        
    n = p * q
    phi = (p - 1) * (q - 1)
    
    e = 3
    while True:
        try:
            d = inverse_mod(e, phi)
            break
        except ValueError:
            e += 2
                
    return (e, n), (d, n)

def encrypt(nb, pub_key):
    e, n = pub_key
    return pow(nb, e, n)

def decrypt(cipher, priv_key):
    d, n = priv_key
    return pow(cipher, d, n)

def prints(s):
    print(f"[SERVEUR] {s}")

def printc(s):
    print(f"[CLIENT] {s}")

def laurent():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(('localhost', PORT))
    s.listen(True)

    prints(f"En attente de connexions sur le port {PORT}...")

    conn1, addr1 = s.accept()
    prints(f"Alice connectée depuis {addr1}")
    
    conn2, addr2 = s.accept()
    prints(f"Bertrand connecté depuis {addr2}")

    # Echange des clés publiques (récupéré par Laurent)
    alice_pub_key = pickle.loads(conn1.recv(4096))
    printc(f"Laurent a reçu la clé publique d'Alice : {alice_pub_key}")
    conn2.send(pickle.dumps(alice_pub_key))
    printc(f"Laurent a envoyé la clé publique d'Alice à Bertrand")

    bert_pub_key = pickle.loads(conn2.recv(4096))
    bert_e, bert_n = bert_pub_key
    printc(f"Laurent a reçu la clé publique de Bertrand : {bert_pub_key}")
    conn1.send(pickle.dumps(bert_pub_key))
    printc(f"Laurent a envoyé la clé publique de Bertrand à Alice")

    while True:
        try:
            data1 = conn1.recv(4096)
            if not data1:
                break
            encrypted_msg1 = pickle.loads(data1)
            factor = pow(2, bert_e, bert_n)
            modified_cipher1 = (encrypted_msg1 * factor) % bert_n
            conn2.send(pickle.dumps(modified_cipher1))
            prints(f"Message modifié d'Alice vers Bertrand ({len(data1)} bytes)")

            data2 = conn2.recv(4096)
            if not data2:
                break
            prints(f"Message relayé de Bertrand vers Alice ({len(data2)} bytes)")
            conn1.send(data2)
        except Exception as e:
            prints(f"Erreur de relais: {e}")
            break
            
    conn1.close()
    conn2.close()
    s.close()

def alice():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect(('localhost', PORT))
    printc(f"Alice connectée à Laurent sur le port {PORT}")

    # Génération des clés d'Alice
    pub_key, priv_key = generate_rsa_keypair()
    printc(f"Alice a généré ses clés (pub: {pub_key})")
    
    # Envoi de la clé publique
    s.send(pickle.dumps(pub_key))
    
    # Réception de la clé publique de Bertrand
    bert_pub_key = pickle.loads(s.recv(4096))
    printc(f"Alice a reçu la clé publique de Bertrand : {bert_pub_key}")
    
    messages_to_send = [
        123456789,
        987654321,
        555555555
    ]

    for msg in messages_to_send:
        # Envoi d'un message chiffré
        encrypted_msg = encrypt(msg, bert_pub_key)
        s.send(pickle.dumps(encrypted_msg))
        printc(f"Alice a envoyé : '{msg}' (chiffré)")
        
        # Réception de la réponse
        encrypted_resp = pickle.loads(s.recv(4096))
        resp = decrypt(encrypted_resp, priv_key)
        printc(f"Alice a reçu et déchiffré : '{resp}'")

    s.close()

def bertrand():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect(('localhost', PORT))
    printc(f"Bertrand connecté à Laurent sur le port {PORT}")

    # Génération des clés de Bertrand
    pub_key, priv_key = generate_rsa_keypair()
    printc(f"Bertrand a généré ses clés (pub: {pub_key})")
    
    # Envoi de la clé publique
    s.send(pickle.dumps(pub_key))
    
    # Réception de la clé publique d'Alice
    alice_pub_key = pickle.loads(s.recv(4096))
    printc(f"Bertrand a reçu la clé publique d'Alice : {alice_pub_key}")
    
    responses = [
        111111111,
        222222222,
        333333333
    ]

    for resp in responses:
        # Attente d'un message chiffrée d'Alice
        encrypted_msg = pickle.loads(s.recv(4096))
        msg = decrypt(encrypted_msg, priv_key)
        printc(f"Bertrand a reçu et déchiffré : '{msg}'")

        # Envoi de la réponse chiffrée
        encrypted_resp = encrypt(resp, alice_pub_key)
        s.send(pickle.dumps(encrypted_resp))
        printc(f"Bertrand a envoyé : '{resp}' (chiffré)")

    s.close()

if __name__ == "__main__":
    import threading
    import time

    print("Démarrage du réseau...")
    
    try:
        # Démarrage de Laurent (serveur) dans un thread
        serveur_thread = threading.Thread(target=laurent, daemon=True)
        serveur_thread.start()

        time.sleep(1) # Laisse le temps au serveur de s'initialiser

        # Démarrage d'Alice (client 1)
        alice_thread = threading.Thread(target=alice)
        alice_thread.start()

        time.sleep(1) # Laisse le temps à Alice de se connecter en premier

        # Démarrage de Bertrand (client 2)
        bertrand_thread = threading.Thread(target=bertrand)
        bertrand_thread.start()

        # Attente de la fin des échanges clients
        alice_thread.join()
        bertrand_thread.join()
    except Exception as e:
        print(f"Erreur dans le réseau: {e}")
        serveur_thread.join(timeout=1)
        alice_thread.join(timeout=1)
        bertrand_thread.join(timeout=1)
    print("Communication terminée.")
